# DNABERT2-E fine-tuning on hard-label training data

This notebook fine-tunes the repository's `EpigenDnabert2` classifier on the hard-label dataset stored in `Data/training_data/TrainingDataWithRejection_hg38_mincpg_4_minlen_10`.

It reuses the existing project code instead of reimplementing the training stack:
- `methyldl.data.dataset.SupervisedDataset` for parquet-backed sequence and CpG methylation loading.
- `methyldl.modelling.classifiers.dnabert2.EpigenDnabert2` for DNABERT2-E with methylation embeddings.
- `methyldl.modelling.classifiers.dnabert2.TrainingArguments` so the trainer setup matches the rest of the repository.

This version uses the DMR attention classification head, with `dmr_label` passed to the model as the contextual DMR identifier for each read.

The target labels are hard labels. In practice this dataset contains 39 biological cell types plus one rejection class.

In [ ]:
from pathlib import Path
import json
import random
import gc

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

from methyldl.data.dataset import SupervisedDataset
from methyldl.modelling.classifiers.dnabert2 import EpigenDnabert2, TrainingArguments

ROOT = Path.cwd().resolve()
if ROOT.name == "EDA":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "Data/training_data/TrainingDataWithRejection_hg38_mincpg_4_minlen_10"
FOUNDATION_MODEL_PATH = ROOT / "foundationalModels/DNABERT-2-117M"
LABELS_DICT_PATH = ROOT / "App/labels_dict.json"
OUTPUT_DIR = ROOT / "output/dnabert2_finetuning_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"
assert FOUNDATION_MODEL_PATH.exists(), f"Missing foundation model directory: {FOUNDATION_MODEL_PATH}"


def has_triton_capable_gpu() -> bool:
    """Return whether the active CUDA device supports Triton attention."""
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability()
    return major >= 8


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_TRITON = has_triton_capable_gpu()

print(f"Project root: {ROOT}")
print(f"Device: {DEVICE}")
print(f"Triton attention enabled: {USE_TRITON}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
SUMMARY_COLUMNS = ["label", "read_length", "dmr_label"]
summary_rows = []
label_series_by_split = {}
dmr_series_by_split = {}

for split in ("train", "valid", "test"):
    frame = pd.read_parquet(DATA_DIR / f"{split}.parquet", columns=SUMMARY_COLUMNS)
    label_series = frame["label"].astype(int)
    dmr_series = frame["dmr_label"].astype(int)
    label_series_by_split[split] = label_series
    dmr_series_by_split[split] = dmr_series
    summary_rows.append(
        {
            "split": split,
            "n_rows": len(frame),
            "n_unique_labels": int(label_series.nunique()),
            "max_label_id": int(label_series.max()),
            "missing_label_ids": sorted(set(range(int(label_series.max()) + 1)) - set(label_series.unique())),
            "n_unique_dmr_labels": int(dmr_series.nunique()),
            "max_dmr_label_id": int(dmr_series.max()),
            "read_length_mean": float(frame["read_length"].mean()),
            "read_length_p95": float(frame["read_length"].quantile(0.95)),
            "read_length_p99": float(frame["read_length"].quantile(0.99)),
            "read_length_max": int(frame["read_length"].max()),
        }
    )

summary_df = pd.DataFrame(summary_rows).set_index("split")
display(summary_df)

with open(LABELS_DICT_PATH) as handle:
    labels_dict = {int(key): value for key, value in json.load(handle).items()}

num_labels = max(int(series.max()) for series in label_series_by_split.values()) + 1
num_dmr_labels = max(int(series.max()) for series in dmr_series_by_split.values()) + 1
if num_labels - 1 not in labels_dict:
    labels_dict[num_labels - 1] = "rejection"

label_names = [labels_dict.get(label_id, f"class_{label_id}") for label_id in range(num_labels)]
label_count_table = pd.DataFrame({
    split: label_series_by_split[split].value_counts().sort_index() for split in label_series_by_split
}).reindex(range(num_labels), fill_value=0)
label_count_table.index.name = "label_id"
label_count_table.insert(0, "label_name", label_names)

dmr_summary_df = pd.DataFrame(
    {
        split: {
            "n_unique_dmr_labels": int(dmr_series_by_split[split].nunique()),
            "max_dmr_label_id": int(dmr_series_by_split[split].max()),
        }
        for split in dmr_series_by_split
    }
).T

display(label_count_table)
display(dmr_summary_df)
print(f"Total number of target classes used for training: {num_labels}")
print(f"Total number of DMR ids reserved by the attention head: {num_dmr_labels}")

## Training configuration

The train and validation splits are used in full by default.

The test split contains more than four million reads, so the notebook evaluates on a stratified sample by default to keep iteration practical. Set `TEST_EVAL_SAMPLE_N = None` later in the notebook if you want to score the full test split.

`lazy_tokenization=True` is used when building datasets so the notebook does not materialize tokenized tensors for every read up front.

In [ ]:
SEED = 8
TRAIN_SAMPLE_N = None
VALID_SAMPLE_N = 10_000
TEST_EVAL_SAMPLE_N = 200_000
MAX_SEQUENCE_LENGTH = 150
LEARNING_RATE = 3e-5
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 16 if DEVICE == "cuda" else 2
PER_DEVICE_EVAL_BATCH_SIZE = 32 if DEVICE == "cuda" else 2
GRADIENT_ACCUMULATION_STEPS = 1
SAVE_STEPS = 1_000
EVAL_STEPS = 50
LOGGING_STEPS = 50
WARMUP_STEPS = 500
SAVE_TOTAL_LIMIT = 2
DMR_LABEL_COLUMN = "dmr_label"
DATASET_COLUMNS = ["seq", "pattern", "label", DMR_LABEL_COLUMN]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def stratified_sample(frame: pd.DataFrame, n_samples: int | None, seed: int = SEED) -> pd.DataFrame:
    """Return a stratified sample of a labeled dataframe."""
    if n_samples is None or n_samples >= len(frame):
        return frame.reset_index(drop=True)
    sampled, _ = train_test_split(
        frame,
        train_size=n_samples,
        stratify=frame["label"],
        random_state=seed,
    )
    return sampled.reset_index(drop=True)


training_args = TrainingArguments(
    run_name="dnabert2_hard_labels_notebook",
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    overwrite_output_dir=True,
    fp16=(DEVICE == "cuda"),
    report_to=[],
    remove_unused_columns=False,
    prediction_loss_only=False,
    skip_memory_metrics=True,
    auto_find_batch_size=False,
)

print(training_args)

In [ ]:
def load_split_frame(split: str, columns: list[str], n_samples: int | None) -> pd.DataFrame:
    """Load one parquet split with optional class-preserving sampling."""
    split_path = DATA_DIR / f"{split}.parquet"
    if n_samples is None:
        return pd.read_parquet(split_path, columns=columns).reset_index(drop=True)

    if split != "test":
        frame = pd.read_parquet(split_path, columns=columns)
        return stratified_sample(frame, n_samples)

    test_counts = label_count_table["test"].fillna(0).astype(int)
    target_counts = ((test_counts / test_counts.sum()) * n_samples).round().astype(int)
    target_counts = target_counts.where(test_counts > 0, 0)
    target_counts = target_counts.clip(upper=test_counts)

    # Per-class rounding usually leaves a small deficit or surplus, so rebalance
    # the residual against the largest available classes until the total matches.
    diff = n_samples - int(target_counts.sum())
    if diff > 0:
        for label_id in target_counts.sort_values(ascending=False).index:
            available = int(test_counts[label_id] - target_counts[label_id])
            if available <= 0:
                continue
            step = min(available, diff)
            target_counts.loc[label_id] += step
            diff -= step
            if diff == 0:
                break
    elif diff < 0:
        for label_id in target_counts.sort_values(ascending=False).index:
            removable = int(target_counts[label_id])
            if removable <= 0:
                continue
            step = min(removable, -diff)
            target_counts.loc[label_id] -= step
            diff += step
            if diff == 0:
                break

    sampled_parts = []
    for label_id, target in target_counts.items():
        target = int(target)
        if target <= 0:
            continue
        # Read one label at a time so a sampled test set can be built without
        # materializing the full multi-million-row test split in memory.
        part = pd.read_parquet(
            split_path,
            columns=columns,
            filters=[("label", "=", int(label_id))],
        )
        if len(part) > target:
            part = part.sample(n=target, random_state=SEED)
        sampled_parts.append(part)

    sampled = pd.concat(sampled_parts, ignore_index=True)
    return sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


model = EpigenDnabert2(
    foundation_model_huggingface=str(FOUNDATION_MODEL_PATH),
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    num_labels=num_labels,
    num_dmr_labels=num_dmr_labels,
    use_cpg_methylation=True,
    use_m6a_methylation=False,
    use_triton=USE_TRITON,
    training_args=training_args,
)

train_df = load_split_frame("train", DATASET_COLUMNS, TRAIN_SAMPLE_N)
valid_df = load_split_frame("valid", DATASET_COLUMNS, VALID_SAMPLE_N)
test_eval_df = load_split_frame("test", DATASET_COLUMNS, TEST_EVAL_SAMPLE_N)

train_dataset = SupervisedDataset(
    data_path_or_list=train_df,
    tokenizer=model.tokenizer,
    data_interface="pandas",
    lazy_tokenization=True,
    include_dmr_ids=True,
    dmr_label_column=DMR_LABEL_COLUMN,
)
valid_dataset = SupervisedDataset(
    data_path_or_list=valid_df,
    tokenizer=model.tokenizer,
    data_interface="pandas",
    lazy_tokenization=True,
    include_dmr_ids=True,
    dmr_label_column=DMR_LABEL_COLUMN,
)
test_eval_dataset = SupervisedDataset(
    data_path_or_list=test_eval_df,
    tokenizer=model.tokenizer,
    data_interface="pandas",
    lazy_tokenization=True,
    include_dmr_ids=True,
    dmr_label_column=DMR_LABEL_COLUMN,
)

display(
    pd.DataFrame(
        [
            {"split": "train", "rows_used": len(train_df)},
            {"split": "valid", "rows_used": len(valid_df)},
            {"split": "test_eval", "rows_used": len(test_eval_df)},
        ]
    )
)

print(f"Using DMR attention head with {num_dmr_labels} reserved DMR ids.")

In [ ]:
gc.collect()
train_result = model.fine_tune(
    training_args=training_args,
    train_dataset=train_dataset,
    val_dataset=valid_dataset,
    test_dataset=test_eval_dataset,
)
train_result

In [ ]:
best_checkpoint = model.trainer.state.best_model_checkpoint
print(f"Best checkpoint: {best_checkpoint}")
train_result

In [ ]:
def top_confusions(confusion: np.ndarray, top_n: int = 15) -> pd.DataFrame:
    """Return the largest off-diagonal entries from a confusion matrix."""
    rows = []
    for true_idx in range(confusion.shape[0]):
        for pred_idx in range(confusion.shape[1]):
            if true_idx == pred_idx:
                continue
            count = int(confusion[true_idx, pred_idx])
            if count == 0:
                continue
            rows.append(
                {
                    "true_label_id": true_idx,
                    "true_label_name": label_names[true_idx],
                    "pred_label_id": pred_idx,
                    "pred_label_name": label_names[pred_idx],
                    "count": count,
                }
            )
    return pd.DataFrame(rows).sort_values("count", ascending=False).head(top_n)


def evaluate_split(split_name: str, dataset: SupervisedDataset):
    """Predict one split and return summary metrics plus error-analysis tables."""
    prediction_output = model.predict(dataset)
    logits = np.asarray(prediction_output.predictions)
    y_true = np.asarray(prediction_output.label_ids).astype(int)

    if logits.ndim == 1:
        probabilities = logits
        y_pred = (probabilities >= 0.5).astype(int)
    else:
        # The model returns raw multi-class logits here, so convert them to
        # probabilities before taking the most likely class for evaluation.
        probabilities = softmax(logits, axis=1)
        y_pred = probabilities.argmax(axis=1)

    metrics = {
        "split": split_name,
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    report = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=list(range(num_labels)),
            target_names=label_names,
            output_dict=True,
            zero_division=0,
        )
    ).T
    confusion = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
    confusion_df = pd.DataFrame(confusion, index=label_names, columns=label_names)

    return metrics, report, confusion_df, top_confusions(confusion)


valid_metrics, valid_report, valid_confusions, valid_top_confusions = evaluate_split("valid", valid_dataset)
test_metrics, test_report, test_confusions, test_top_confusions = evaluate_split("test", test_eval_dataset)

display(pd.DataFrame([valid_metrics, test_metrics]).set_index("split"))
display(valid_top_confusions)
display(test_top_confusions)

In [ ]:
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame([valid_metrics, test_metrics]).to_csv(ARTIFACT_DIR / "metrics.csv", index=False)
valid_report.to_csv(ARTIFACT_DIR / "valid_classification_report.csv")
test_report.to_csv(ARTIFACT_DIR / "test_classification_report.csv")
valid_confusions.to_csv(ARTIFACT_DIR / "valid_confusion_matrix.csv")
test_confusions.to_csv(ARTIFACT_DIR / "test_confusion_matrix.csv")

with open(ARTIFACT_DIR / "label_names.json", "w") as handle:
    json.dump({label_id: label_name for label_id, label_name in enumerate(label_names)}, handle, indent=2)

print(f"Saved notebook artifacts to: {ARTIFACT_DIR}")